In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from russian_rates.analysis import align_ruonia_with_key_rate
from russian_rates.models import (
    prepare_transitions,
    calibrate_profile,
    conditional_moments,
)

In [2]:
ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "pyproject.toml").is_file()
)

data_dir = ROOT / "data" / "processed"

ruonia = pd.read_parquet(data_dir / "ruonia.parquet")
key_rate = pd.read_parquet(data_dir / "key_rate.parquet")

df = align_ruonia_with_key_rate(ruonia, key_rate)
df = df.dropna(
    subset=["ruonia_pct", "key_rate_pct", "spread_pp"]
).sort_values("date").reset_index(drop=True)

train = df[df["date"] >= "2023-01-01"].copy()

r0, r1, dt = prepare_transitions(train, "spread_pp")

fit = calibrate_profile(
    r0, r1, dt,
    a_bounds=(0.001, 200.0),
)

a, b, sigma = fit["a"], fit["b"], fit["sigma"]

s0 = df["spread_pp"].iloc[-1] / 100
key0 = float(df["key_rate_pct"].iloc[-1])
start_date = pd.Timestamp(df["date"].iloc[-1])

print(f"a = {a:.4f}")
print(f"b = {b * 100:.4f} п.п.")
print(f"sigma = {sigma * 100:.4f} п.п./√год")
print(f"Начальная ключевая ставка = {key0:.2f}%")
print(f"Дата начала = {start_date.date()}")

a = 42.8224
b = -0.2113 п.п.
sigma = 2.9518 п.п./√год
Начальная ключевая ставка = 14.00%
Дата начала = 2026-11-20


In [3]:
horizons = [30, 90, 180]
max_days = max(horizons)

days = np.arange(max_days + 1)

key_scenarios = {
    "Без изменений": np.full(max_days + 1, key0),

    "Повышение +2 п.п.": (
        key0 + 2 * (days >= 30)
    ),

    "Снижение −2 п.п.": (
        key0 - 2 * (days >= 30)
    ),
}

# Даты 1..180 — будущие календарные дни
future_dates = pd.date_range(
    start=start_date + pd.Timedelta(days=1),
    periods=max_days,
    freq="D",
)

# Контрольные проверки
assert key_scenarios["Повышение +2 п.п."][29] == key0
assert key_scenarios["Повышение +2 п.п."][30] == key0 + 2

for h in horizons:
    avg_key = key_scenarios["Повышение +2 п.п."][1:h+1].mean()

    print(f"{h} дней: средняя ключевая = {avg_key:.4f}%")

30 дней: средняя ключевая = 14.0667%
90 дней: средняя ключевая = 15.3556%
180 дней: средняя ключевая = 15.6778%


C:\Users\bekishev\AppData\Local\Temp\ipykernel_6888\1411933052.py:20: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  start=start_date + pd.Timedelta(days=1),


In [4]:
n_paths = 10_000
rng = np.random.default_rng(42)

# Строка 0 — последнее известное состояние.
# Строки 1..180 — будущие дневные состояния.
spread_paths = np.empty((max_days + 1, n_paths))
spread_paths[0, :] = s0

for t in range(1, max_days + 1):
    mu, var = conditional_moments(
        a, b, sigma,
        spread_paths[t-1],
        1 / 365,
    )

    spread_paths[t] = (
        mu + np.sqrt(var) * rng.standard_normal(n_paths)
    )

# Переводим в процентные пункты
spread_paths_pp = spread_paths * 100

print("Размер массива:", spread_paths_pp.shape)

Размер массива: (181, 10000)


In [5]:
results = []

# Аналитическое ожидание спреда на каждый будущий день
expected_spread = (
    b + (s0 - b) * np.exp(-a * days[1:] / 365)
) * 100

for scenario, key_path in key_scenarios.items():

    # RUONIA = ключевая ставка + спред
    ruonia_paths = (
        key_path[:, None] + spread_paths_pp
    )

    for h in horizons:

        # Учитываем ровно h будущих календарных дней
        average_rates = (
            ruonia_paths[1:h+1].mean(axis=0)
        )

        # Независимая аналитическая проверка среднего
        expected_average = (
            key_path[1:h+1].mean()
            + expected_spread[:h].mean()
        )

        results.append({
            "Сценарий": scenario,
            "Дней": h,
            "Аналитическое среднее": expected_average,
            "Среднее MC": average_rates.mean(),
            "Std средней": average_rates.std(ddof=1),
            "5%": np.quantile(average_rates, 0.05),
            "Медиана": np.median(average_rates),
            "95%": np.quantile(average_rates, 0.95),
        })

average_results = pd.DataFrame(results)

display(average_results.round(4))

,Сценарий,Дней,Аналитическое среднее,Среднее MC,Std средней,5%,Медиана,95%
0,Без изменений,30,13.8618,13.8614,0.1860,13.5593,13.8600,14.1708
1,Без изменений,90,13.8138,13.8129,0.1293,13.5991,13.8125,14.0240
2,Без изменений,180,13.8012,13.8021,0.0943,13.6466,13.8025,13.9583
3,Повышение +2 п.п.,30,13.9284,13.9281,0.1860,13.6259,13.9267,14.2375
4,Повышение +2 п.п.,90,15.1693,15.1684,0.1293,14.9547,15.1681,15.3796
5,Повышение +2 п.п.,180,15.4790,15.4799,0.0943,15.3244,15.4803,15.6361
6,Снижение −2 п.п.,30,13.7951,13.7948,0.1860,13.4926,13.7933,14.1041
7,Снижение −2 п.п.,90,12.4582,12.4573,0.1293,12.2436,12.4570,12.6685
8,Снижение −2 п.п.,180,12.1234,12.1243,0.0943,11.9689,12.1248,12.2805
